# 构建 Micrograd - 练习

来自 [构建 Micrograd 视频](https://www.youtube.com/watch?v=VMj-3S1tku0) 的练习。<br>
本笔记本是 [原始 micrograd 练习笔记本](https://colab.research.google.com/drive/1FPTx1RXtBfc4MaTkf7viZZD4U2F9gtKN?usp=sharing) 的稍作重新格式化的版本。

1. 在 YouTube 上观看 [构建 Micrograd 视频](https://www.youtube.com/watch?v=VMj-3S1tku0)
2. 回来完成这些练习以提升等级 :)

## 第一部分：导数

这是一个接收 $3$ 个输入并产生一个输出的数学表达式：

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from math import sin, cos, exp, log

In [ ]:
def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5 # assuming c != 0, a >= 0

print(f(2, 3, 4))

编写函数 `gradf`，返回 $f$ 的解析梯度 (analytical gradient)，即利用你的微积分技巧求导，然后实现公式。<br>
如果你不懂微积分，可以随时询问 WolframAlpha，例如：https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

In [ ]:
def gradf(a, b, c):
  # --------------------
  # TODO: Your code here to return [df/da, df/db, df/dc]
  # --------------------
  return [0, 0, 0]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")

现在在不使用任何微积分的情况下，使用我们在视频中使用的近似方法来数值估计梯度。<br>
你不应该调用上一个单元格中的函数 `df`。

In [ ]:
# -----------
# TODO: Your code here
numerical_grad = [0, 0, 0]
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")

还有另一种公式可以为函数的导数提供更好的数值近似。<br><br>
在此处了解它：https://en.wikipedia.org/wiki/Symmetric_derivative

**实现它。**<br><br>
确认对于 *相同的步长 $h$*，此版本提供了更好的近似。

In [ ]:
# -----------
# TODO: Your code here
numerical_grad2 = [0, 0, 0]
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")

## 第二部分：支持 Softmax

在不过多参考代码/视频的情况下，使以下两个单元格正常工作。

你必须实现（在某些情况下是重新实现）Value 对象的许多函数，类似于我们在视频中看到的。<br>
这里实现的不是平方误差损失，而是**负对数似然损失 (negative log likelihood loss)**，这在分类中非常常用。

In [ ]:
# Value class starter code, with many functions taken out
class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  # ------
  # re-implement all the other functions needed for the exercises below
  # TODO: Your code here
  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts)
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")

**验证：** 使用 `torch` 库计算出的梯度应该完全相同。

In [ ]:
import torch

# TODO: Your code here

<b>恭喜！你升级了！</b>